# 02 XGBoost Weekend Model (Anchored Walk-Forward)

This notebook trains leakage-safe XGBoost regressors for weekend return prediction.

- Target: `ret_weekend_close`
- CV: anchored walk-forward folds
- Tuning: Optuna with fold-internal validation split and early stopping


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import optuna
import shap
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUT_DIR = PROJECT_ROOT / "research_outputs" / "weekend_rl_xgb"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
MODEL_DIR = OUT_DIR / "models"
for p in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

panel = pd.read_parquet(OUT_DIR / "decision_panel.parquet")
with open(OUT_DIR / "decision_panel_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

feature_cols = metadata["feature_cols"]
panel = panel.sort_values(["date_decision", "ticker"]).reset_index(drop=True)
panel.head()


In [ ]:
def make_anchored_folds(df: pd.DataFrame, n_folds: int = 5, min_train_years: int = 5):
    dates = np.array(sorted(df["date_decision"].unique()))
    start_date = pd.Timestamp(dates.min())
    fold_span = (pd.Timestamp(dates.max()) - pd.Timestamp(dates.min())).days / (n_folds + 1)
    folds = []

    for i in range(n_folds):
        train_end = start_date + pd.Timedelta(days=int((min_train_years * 365) + i * fold_span))
        val_end = train_end + pd.Timedelta(days=int(0.5 * fold_span))
        test_end = val_end + pd.Timedelta(days=int(0.5 * fold_span))

        train_idx = df["date_decision"] <= train_end
        val_idx = (df["date_decision"] > train_end) & (df["date_decision"] <= val_end)
        test_idx = (df["date_decision"] > val_end) & (df["date_decision"] <= test_end)

        if train_idx.sum() == 0 or val_idx.sum() == 0 or test_idx.sum() == 0:
            continue
        folds.append((train_idx, val_idx, test_idx))

    return folds

folds = make_anchored_folds(panel, n_folds=6, min_train_years=4)
print("n_folds", len(folds))
for i, (tr, va, te) in enumerate(folds, start=1):
    print(i, panel.loc[tr, "date_decision"].min(), panel.loc[tr, "date_decision"].max(),
          panel.loc[va, "date_decision"].min(), panel.loc[va, "date_decision"].max(),
          panel.loc[te, "date_decision"].min(), panel.loc[te, "date_decision"].max())


In [ ]:
def directional_accuracy(y_true, y_pred):
    return (np.sign(y_true) == np.sign(y_pred)).mean()


def tune_xgb_optuna(X_train, y_train, X_val, y_val, n_trials=25):
    def objective(trial):
        params = {
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 10.0),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
            "objective": "reg:squarederror",
            "random_state": SEED,
            "tree_method": "hist",
        }

        model = xgb.XGBRegressor(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        pred_val = model.predict(X_val)
        return mean_squared_error(y_val, pred_val)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

results = []
pred_parts = []
best_params_by_fold = {}

for fold_id, (train_idx, val_idx, test_idx) in enumerate(folds, start=1):
    df_train = panel.loc[train_idx].copy()
    df_val = panel.loc[val_idx].copy()
    df_test = panel.loc[test_idx].copy()

    scaler = StandardScaler()
    X_train = scaler.fit_transform(df_train[feature_cols])
    X_val = scaler.transform(df_val[feature_cols])
    X_test = scaler.transform(df_test[feature_cols])

    y_train = df_train["ret_weekend_close"].values
    y_val = df_val["ret_weekend_close"].values
    y_test = df_test["ret_weekend_close"].values

    best_params = tune_xgb_optuna(X_train, y_train, X_val, y_val, n_trials=20)
    best_params.update({"objective": "reg:squarederror", "random_state": SEED, "tree_method": "hist"})

    model = xgb.XGBRegressor(**best_params)
    X_trval = np.vstack([X_train, X_val])
    y_trval = np.concatenate([y_train, y_val])
    model.fit(X_trval, y_trval, verbose=False)

    y_pred = model.predict(X_test)
    fold_metrics = {
        "fold_id": fold_id,
        "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "mae": float(mean_absolute_error(y_test, y_pred)),
        "r2_oos": float(r2_score(y_test, y_pred)),
        "dir_acc": float(directional_accuracy(y_test, y_pred)),
    }
    results.append(fold_metrics)
    best_params_by_fold[fold_id] = best_params

    out = df_test[["date_decision", "ticker", "ret_weekend_close"]].copy()
    out["y_pred_xgb"] = y_pred
    out["recent_realized_vol"] = df_test["vol_21"].values
    out["score_risk_adj"] = out["y_pred_xgb"] / (out["recent_realized_vol"].abs() + 1e-8)
    out["fold_id"] = fold_id
    out["split_label"] = "test"
    pred_parts.append(out)

metrics_df = pd.DataFrame(results)
xgb_predictions_panel = pd.concat(pred_parts, ignore_index=True).sort_values(["date_decision", "ticker"])

metrics_df


In [ ]:
# Sign bucket diagnostics.
xgb_predictions_panel["sign_true"] = np.sign(xgb_predictions_panel["ret_weekend_close"])
xgb_predictions_panel["sign_pred"] = np.sign(xgb_predictions_panel["y_pred_xgb"])
by_sign = (
    xgb_predictions_panel.groupby("sign_true")
    .agg(
        n=("ret_weekend_close", "size"),
        mean_true=("ret_weekend_close", "mean"),
        mean_pred=("y_pred_xgb", "mean"),
        hit=("sign_pred", lambda s: np.nan),
    )
)

# Hit by sign bucket computed manually.
rows = []
for sgn, grp in xgb_predictions_panel.groupby("sign_true"):
    rows.append({
        "sign_true": sgn,
        "n": len(grp),
        "mean_true": grp["ret_weekend_close"].mean(),
        "mean_pred": grp["y_pred_xgb"].mean(),
        "hit_rate": (grp["sign_true"] == grp["sign_pred"]).mean(),
    })
sign_bucket_df = pd.DataFrame(rows).sort_values("sign_true")
sign_bucket_df


In [ ]:
# Fit final model on all rows except final period for explainability snapshot.
# Use last 20% as holdout for simple SHAP diagnostic.
cutoff = panel["date_decision"].quantile(0.8)
tr = panel["date_decision"] <= cutoff
te = panel["date_decision"] > cutoff

scaler_final = StandardScaler()
X_tr = scaler_final.fit_transform(panel.loc[tr, feature_cols])
X_te = scaler_final.transform(panel.loc[te, feature_cols])
y_tr = panel.loc[tr, "ret_weekend_close"].values

final_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    random_state=SEED,
    tree_method="hist",
    max_depth=4,
    min_child_weight=5.0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.01,
    reg_lambda=1.0,
    gamma=0.0,
    learning_rate=0.05,
    n_estimators=500,
)
final_model.fit(X_tr, y_tr, verbose=False)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_te[: min(400, len(X_te))])
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, pd.DataFrame(X_te[: min(400, len(X_te))], columns=feature_cols), show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "xgb_shap_summary.png", dpi=150)
plt.close()


In [ ]:
# Persist interfaces.
xgb_predictions_panel = xgb_predictions_panel.rename(columns={"ret_weekend_close": "y_true_weekend"})

# Attach selected feature set for RL state reconstruction.
merge_cols = ["date_decision", "ticker"] + feature_cols
xgb_predictions_panel = xgb_predictions_panel.merge(panel[merge_cols], on=["date_decision", "ticker"], how="left")

xgb_predictions_panel.to_parquet(OUT_DIR / "xgb_predictions_panel.parquet", index=False)
metrics_df.to_csv(TABLE_DIR / "xgb_fold_metrics.csv", index=False)
sign_bucket_df.to_csv(TABLE_DIR / "xgb_sign_bucket_metrics.csv", index=False)
with open(OUT_DIR / "xgb_best_params_by_fold.json", "w", encoding="utf-8") as f:
    json.dump(best_params_by_fold, f, indent=2)

print(metrics_df.describe(include="all"))
print("saved:", OUT_DIR / "xgb_predictions_panel.parquet")


In [ ]:
# Validation checks.
assert xgb_predictions_panel["date_decision"].is_monotonic_increasing, "Predictions not date-sorted"
assert set(["date_decision", "ticker", "y_true_weekend", "y_pred_xgb", "recent_realized_vol", "score_risk_adj", "fold_id", "split_label"]).issubset(xgb_predictions_panel.columns)
print("xgb_predictions_panel schema check: OK")
